In [22]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
import langgraph
from langchain_core.messages import HumanMessage, SystemMessage

# Load .env file
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

AGENT_SUBFOLDER = "project_files"


In [23]:
from typing import Annotated, List, TypedDict
import operator
from pydantic import BaseModel, Field
from langgraph.types import Send

# Schema for structured output to use in planning
class File(BaseModel):
    file_name: str = Field(
        description="Name for this file in the project.",
    )
    description: str = Field(
        description="A brief description of everything that needs to be in this file, including functions and logic. Make the description as concise as possible and be decisive about the features.",
    )
    packages: str = Field(
        description="A list of the packages and technologies that should be used in the development of this file. Mention ONLY the packages by name and no other details.",
    )

class Files(BaseModel):
    files: List[File] = Field(
        description="Files in the project.",
    )
    file_structure: str = Field(
        description="The file organization pattern."
    )

# Augment the LLM with schema for structured output
planner = llm.with_structured_output(Files)

In [30]:
import os, re
import subprocess
import shlex
import platform
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import StructuredTool
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# ===== Configuration =====
SANDBOX_DIR = os.getcwd()  # Restrict all operations here
MAX_FILE_SIZE = 5 * 1024 * 1024  # 5 MB limit
TIMEOUT = 5  # 5 seconds timeout for shell commands

# ===== 1. Safe Bash Tool =====
class BashInputSchema(BaseModel):
    command: str = Field(..., description="The shell command to execute (safe mode)")


def run_safe_shell(command: str) -> str:
    forbidden_keywords = ["rm", "mv", "sudo", "shutdown", "reboot", ">", ">>", ":", "|", "&", ";", "`", "$(", "curl", "wget"]
    if any(kw in command for kw in forbidden_keywords):
        return "Error: Command contains forbidden operations."
    if ".." in command or command.startswith("/") or command.startswith("\\"):
        return "Error: Access outside the sandbox directory is not allowed."

    try:
        if platform.system() == "Windows":
            # Use cmd.exe or PowerShell
            result = subprocess.run(
                ["cmd.exe", "/C", command],   # or ["powershell", "-Command", command]
                capture_output=True,
                text=True,
                cwd=SANDBOX_DIR,
                timeout=TIMEOUT
            )
        else:
            # Use bash on Unix-like systems
            result = subprocess.run(
                shlex.split(command),
                capture_output=True,
                text=True,
                cwd=SANDBOX_DIR,
                timeout=TIMEOUT
            )
        
        return result.stdout.strip() if result.returncode == 0 else f"Error:\n{result.stderr.strip()}"
    except subprocess.TimeoutExpired:
        return f"Error: Command timed out after {TIMEOUT} seconds."
    except Exception as e:
        return f"Error: {str(e)}"


safe_shell_tool = StructuredTool.from_function(
    func=run_safe_shell,
    name="safe_shell",
    description="Execute a safe shell command (restricted to current directory, no dangerous commands, 5s timeout). Works on Windows and Unix.",
    args_schema=BashInputSchema
)

# ===== 2. File Read Tool =====
class FileReadInput(BaseModel):
    file_path: str = Field(..., description="Path to the file (relative to sandbox)")

def read_file(file_path: str) -> str:
    try:
        abs_path = os.path.abspath(os.path.join(SANDBOX_DIR, file_path))
        if not abs_path.startswith(SANDBOX_DIR):
            return "Error: Access outside the sandbox directory is not allowed."
        if os.path.isdir(abs_path):
            return "Error: Path is a directory, not a file."
        if not os.path.exists(abs_path):
            return "Error: File does not exist."
        if os.path.getsize(abs_path) > MAX_FILE_SIZE:
            return f"Error: File exceeds maximum size of {MAX_FILE_SIZE // (1024*1024)} MB."
        with open(abs_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception as e:
        return f"Error: {str(e)}"

file_read_tool = StructuredTool.from_function(
    func=read_file,
    name="read_file",
    description="Read the contents of a file from the current directory.",
    args_schema=FileReadInput
)

# ===== 3. File List Tool =====
class FileListInput(BaseModel):
    directory: str = Field(".", description="Directory path relative to sandbox")

def list_files(directory: str = f'./{AGENT_SUBFOLDER}') -> str:
    try:
        abs_path = os.path.abspath(os.path.join(SANDBOX_DIR, directory))

        # Ensure the path stays within SANDBOX_DIR
        if os.path.commonpath([SANDBOX_DIR, abs_path]) != SANDBOX_DIR:
            return "Error: Access outside the sandbox directory is not allowed."

        if not os.path.exists(abs_path):
            return "Error: Directory does not exist."
        if not os.path.isdir(abs_path):
            return "Error: Path is not a directory."

        files = os.listdir(abs_path)
        return "\n".join(files) if files else "Directory is empty."
    except Exception as e:
        return f"Error: {str(e)}"

file_list_tool = StructuredTool.from_function(
    func=list_files,
    name="list_files",
    description="List all files in a given directory (restricted to sandbox and its subfolders).",
    args_schema=FileListInput
)

# ===== 4. File Edit Tool =====
class FileEditInput(BaseModel):
    file_path: str = Field(..., description="Path to the file (relative to sandbox)")
    content: str = Field(..., description="Content to write to the file")
    mode: Literal["overwrite", "append"] = Field("overwrite", description="Overwrite or append to the file")

def edit_file(file_path: str, content: str, mode: str = "overwrite") -> str:
    try:
        abs_path = os.path.abspath(os.path.join(SANDBOX_DIR, file_path))
        if not abs_path.startswith(SANDBOX_DIR):
            return "Error: Access outside the sandbox directory is not allowed."
        if any(part.startswith('.') for part in file_path.split(os.sep)):
            return "Error: Editing hidden/system files is not allowed."
        if os.path.exists(abs_path) and os.path.getsize(abs_path) > MAX_FILE_SIZE:
            return f"Error: File exceeds maximum allowed size of {MAX_FILE_SIZE // (1024*1024)} MB."
        os.makedirs(os.path.dirname(abs_path), exist_ok=True)
        with open(abs_path, 'a' if mode == "append" else 'w', encoding="utf-8") as f:
            f.write(content)
        return f"Success: File '{file_path}' {'appended' if mode == 'append' else 'overwritten'} successfully."
    except Exception as e:
        return f"Error: {str(e)}"

file_edit_tool = StructuredTool.from_function(
    func=edit_file,
    name="edit_file",
    description="Safely edit a file by overwriting or appending text (restricted to current directory).",
    args_schema=FileEditInput
)

# ===== 5. Code Search Tool (using ripgrep) =====
class CodeSearchInput(BaseModel):
    pattern: str = Field(..., description="The search pattern (regex supported)")
    directory: str = Field(".", description="Directory to search in (relative to sandbox)")

def code_search(pattern: str, directory: str = ".") -> str:
    try:
        abs_dir = os.path.abspath(os.path.join(SANDBOX_DIR, directory))
        
        # Ensure search stays inside sandbox
        if os.path.commonpath([SANDBOX_DIR, abs_dir]) != SANDBOX_DIR:
            return "Error: Access outside the sandbox directory is not allowed."
        if not os.path.exists(abs_dir):
            return "Error: Directory does not exist."

        matches = []
        # Walk through all subdirectories
        for root, _, files in os.walk(abs_dir):
            for file in files:
                path = os.path.join(root, file)
                # Only process text/code files
                if os.path.isfile(path):
                    try:
                        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                            for i, line in enumerate(f, 1):
                                if re.search(pattern, line):
                                    matches.append(f"{os.path.relpath(path, SANDBOX_DIR)}:{i}:{line.strip()}")
                    except Exception as e:
                        matches.append(f"Error reading {file}: {str(e)}")

        return "\n".join(matches) if matches else ""
    except Exception as e:
        return f"Error: {str(e)}"

code_search_tool = StructuredTool.from_function(
    func=code_search,
    name="code_search",
    description="Search for code patterns using regex in the sandbox directory and subfolders. Works on Windows and Unix.",
    args_schema=CodeSearchInput
)

# ===== All tools in a list =====
all_tools = [safe_shell_tool, file_read_tool, file_list_tool, file_edit_tool, code_search_tool]
coder_tools = [file_read_tool, file_list_tool, file_edit_tool, code_search_tool]

coder_llm = llm.bind_tools(coder_tools)
tester_llm = llm.bind_tools(all_tools)

In [31]:
tool_map = {
    "edit_file": file_edit_tool,
    "read_file": file_read_tool,
    "list_files": file_list_tool,
    "code_search": code_search_tool,
    "safe_shell": safe_shell_tool
}

import json

def run_tool(llm_response):
    if hasattr(llm_response, "tool_calls") and llm_response.tool_calls:
        for call in llm_response.tool_calls:
            print(call)
            tool_name = call["name"]
            args = call["args"]
            # If args is a string, parse it
            '''if isinstance(args, str):
                try:
                    args = json.loads(args)
                except Exception:
                    pass  # leave as is if not JSON'''
            tool = tool_map.get(tool_name)
            if tool:
                try:
                    result = tool.invoke(args)
                except TypeError as e:
                    print(f"TypeError: {e}")
                    print("Args:", args)
                    # Optionally, try passing as a single dict if needed
                print(f"{tool_name} result:", result)
            else:
                print(f"Unknown tool: {tool_name}")
    else:
        print("No tool calls found. LLM content:", getattr(llm_response, "content", llm_response))

In [33]:
# Graph state
class State(TypedDict):
    project: str
    files: list[Files]
    completed_files: Annotated[list, operator.add]
    test_results: Annotated[list, operator.add]
    retries_left: Annotated[list, operator.add]
    final_project: str

class WorkerState(TypedDict):
    file: File
    completed_files: Annotated[list, operator.add]
    test_results: Annotated[list, operator.add]
    retries_left: Annotated[list, operator.add]


# Nodes
def orchestrator(state: State):
    """Orchestrator that generates a plan for the project"""

    # Generate queries
    project_files = planner.invoke(
        [
            SystemMessage(content="You are a project manager and senior developer. Generate a plan for the given project. Create a list of files required for the project, descriptions for each of the files, and the required packages for each file. Also include the required file organization."),
            HumanMessage(content=f"Here is a description of the project: {state['project']}"),
        ]
    )

    return {"files": project_files.files}


def llm_call(state: WorkerState): 
    """Worker writes one file of the project, generates unit tests,
    retries fixing code until tests pass, and regenerates tests once if needed."""

    file_name = state["file"].file_name
    description = state["file"].description
    packages = state["file"].packages

    file_path = os.path.join(AGENT_SUBFOLDER, file_name)
    retries = state.get("retries_left", 3)

    # Step 1: Generate code once
    code = coder_llm.invoke(
        [
            SystemMessage(
                content="You are a software developer. Write only the code for the requested file, without any headers, backticks or other symbols."
            ),
            HumanMessage(
                content=f"File name: {file_path}\nDescription: {description}\nPackages: {packages}"
            ),
        ]
    )
    run_tool(code)
    # Read code back for test generation
    code_content = file_read_tool.invoke({"file_path": file_path})

    # Step 2: Generate tests once
    test_file = f"test_{file_name.replace('.py', '')}.py"
    test_file_path = os.path.join(AGENT_SUBFOLDER, test_file)
    test_code = coder_llm.invoke(
        [
            SystemMessage(
                content="You are a software developer. Write pytest unit tests for the given Python file. "
                        "Ensure tests import the correct functions/classes. Only output the test code."
            ),
            HumanMessage(content=f"Here is the implementation to test:\n\n{code_content}")
        ]
    )
    run_tool(test_code)

    # Flag for whether we already retried test generation
    test_regenerated = False

    # Step 3: Retry loop only for testing and fixing implementation
    while retries > 0:
        test_result = safe_shell_tool.invoke({"command": f"pytest {test_file_path} -q"})

        if "Error" not in test_result and "failed" not in test_result.lower():
            # Success
            return {
                "completed_files": [{"file_name": file_name, "content": code}],
                "test_results": [test_result],
                "retries_left": [retries]
            }

        # If the problem is in the tests themselves (syntax/import issues), regenerate once
        if not test_regenerated and ("ImportError" in test_result or "SyntaxError" in test_result):
            test_code = coder_llm.invoke(
                [
                    SystemMessage(
                        content="You are a software developer. The earlier tests had issues. "
                                "Please regenerate correct pytest unit tests for the given code."
                    ),
                    HumanMessage(content=f"Here is the implementation to test:\n\n{code_content}")
                ]
            )
            run_tool(test_code)

            test_regenerated = True
            continue  # rerun tests without consuming a retry

        # Otherwise → try to fix implementation
        error_output = test_result
        fix_code = coder_llm.invoke(
            [
                SystemMessage(
                    content="You are a software developer. Fix the following code based on test failures. "
                            "Only output the fixed code."
                ),
                HumanMessage(
                    content=f"Original file: {file_name}\nErrors:\n{error_output}\nFix the code."
                ),
            ]
        )

        # Save fixed code (overwrite)

        run_tool(fix_code)
        code = fix_code  # update working code
        code_content = fix_code  # update for debugging/tests if needed
        retries -= 1

    # If retries exhausted
    return {
        "completed_files": [{"file_name": file_name, "content": code}],
        "test_results": ["Tests did not pass after retries."],
        "retries_left": [retries]
    }


def synthesizer(state: State):
    print(state)
    return state


# Conditional edge function to create llm_call workers that each write a section of the report
def assign_workers(state: State):
    """Assign a worker to each file in the plan"""
    # Kick off file writing in parallel via Send() API
    return [
        Send("llm_call", {
            "file": file_obj,
            "completed_files": [],
            "retries_left": 3,
            "test_results": ""
        })
        for file_obj in state["files"]
    ]


# Build workflow
orchestrator_worker_builder = StateGraph(State)

# Add the nodes
orchestrator_worker_builder.add_node("orchestrator", orchestrator)
orchestrator_worker_builder.add_node("llm_call", llm_call)
orchestrator_worker_builder.add_node("synthesizer", synthesizer)

# Add edges to connect nodes
orchestrator_worker_builder.add_edge(START, "orchestrator")
orchestrator_worker_builder.add_conditional_edges(
    "orchestrator", assign_workers, ["llm_call"]
)
orchestrator_worker_builder.add_edge("llm_call", "synthesizer")
orchestrator_worker_builder.add_edge("synthesizer", END)

# Compile the workflow
orchestrator_worker = orchestrator_worker_builder.compile()

# Show the workflow
#display(Image(orchestrator_worker.get_graph().draw_mermaid_png()))

# Invoke
state = orchestrator_worker.invoke({"project": "Create a fully featured clone of twitter."})

from IPython.display import Markdown
#Markdown(state["final_project"])

{'name': 'edit_file', 'args': {'mode': 'overwrite', 'file_path': 'project_files/backend/config.py', 'content': 'import os\nfrom dotenv import load_dotenv\n\nload_dotenv()\n\nclass Config:\n    SQLALCHEMY_DATABASE_URI = os.getenv("DATABASE_URL", "sqlite:///site.db")\n    SECRET_KEY = os.getenv("SECRET_KEY", "a_very_secret_key")\n    # Add other configuration variables here\n'}, 'id': '966b08c4-6a82-4454-a780-4c6777ea17d1', 'type': 'tool_call'}
edit_file result: Success: File 'project_files/backend/config.py' overwritten successfully.
{'name': 'edit_file', 'args': {'file_path': 'project_files/frontend/src/index.js', 'mode': 'overwrite', 'content': "import React from 'react';\nimport ReactDOM from 'react-dom/client';\nimport './index.css';\nimport App from './App';\nimport reportWebVitals from './reportWebVitals';\n\nconst root = ReactDOM.createRoot(document.getElementById('root'));\nroot.render(\n  <React.StrictMode>\n    <App />\n  </React.StrictMode>\n);\n\n// If you want to start meas